# 🌿 Darukaa Reference Benchmarking Pipeline

**Compare any biodiversity indicator against ecoregion-specific reference benchmarks.**

This notebook runs the full pipeline:
1. Clone the repo & install dependencies
2. Authenticate Google Earth Engine
3. Upload your KML site file
4. Run Tier 1 (global) + Tier 2 (contemporary) benchmarking
5. Download the scorecard (JSON + CSV)

**Methodology:**
- McElderry et al. (2024) DOI:10.32942/X2689N — SEED reference selection
- McNellie et al. (2020) DOI:10.1111/gcb.15383 — Contemporary reference states
- Yen et al. (2019) DOI:10.1002/eap.1970 — Biodiversity benchmarks

---

## 1. Setup — Clone repo & install

In [17]:
# ── Setup: Clone or update repo ──────────────────────────
import os
os.chdir("/content")  # Always start from a safe location

REPO_DIR = "/content/reference-benchmarking"
CODE_DIR = f"{REPO_DIR}/darukaa_reference_v0.1.0"

if os.path.exists(REPO_DIR):
    os.chdir(CODE_DIR)
    !git pull origin main
    print("✓ Pulled latest changes")
else:
    !git clone https://@github.com/G-auravSingh/reference-benchmarking.git {REPO_DIR}
    os.chdir(CODE_DIR)
    print("✓ Cloned fresh")

!pip install -e . --force-reinstall -q 2>&1 | tail -1

import sys
cleared = [k for k in sys.modules if 'darukaa' in k]
for m in cleared:
    del sys.modules[m]

os.chdir("/content")
print(f"✓ Installed from {CODE_DIR}, cleared {len(cleared)} cached modules")

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 5 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 1.10 KiB | 561.00 KiB/s, done.
From https://github.com/G-auravSingh/reference-benchmarking
 * branch            main       -> FETCH_HEAD
   7602db8..388163c  main       -> origin/main
Updating 7602db8..388163c
Fast-forward
 darukaa_reference_v0.1.0/darukaa_reference/config.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
✓ Pulled latest changes
google-cloud-bigtable 2.35.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
✓ Installed from /content/reference-benchmarking/darukaa_reference_v0.1.0, cleared 10 cached modules


## 2. Authenticate Google Earth Engine

Run the cell below — it will show a link. Click it, sign in with your Google account, copy the token back.

In [18]:
import ee

# Colab has a built-in GEE authenticator
ee.Authenticate()

# Initialize with your GEE project
# Replace with your actual project ID
GEE_PROJECT = "gaurav-singh-007"  # @param {type:"string"}

ee.Initialize(project=GEE_PROJECT)
print(f"✓ GEE initialised with project: {GEE_PROJECT}")

✓ GEE initialised with project: gaurav-singh-007


## 3. Upload your KML file

Run the cell below to upload your project site KML/KMZ/GeoJSON file.

In [3]:
from google.colab import files

print("Upload your KML/KMZ/GeoJSON file:")
uploaded = files.upload()

# Get the filename
SITE_FILE = list(uploaded.keys())[0]
print(f"\n✓ Uploaded: {SITE_FILE} ({len(uploaded[SITE_FILE])/1024:.1f} KB)")

Upload your KML/KMZ/GeoJSON file:


Saving Paglam_CR_Demarcation2025.kmz to Paglam_CR_Demarcation2025.kmz

✓ Uploaded: Paglam_CR_Demarcation2025.kmz (1.6 KB)


## 4. Configure the pipeline

Adjust parameters below as needed.

In [19]:
# ── Configuration ─────────────────────────────────────────

# Which indicators to run?
# Options: "ndvi", "lst", "bii", "eii", "ghm", "msa_globio4", "seed"
# Leave empty for ALL GEE-based indicators
INDICATORS = []  # @param

# Tier 2 reference selection
BUFFER_KM = 100        # Search radius for reference patches (km)
HMI_PERCENTILE = 5     # Top N% least disturbed = reference

# NDVI year
NDVI_YEAR = 2025       # @param {type:"integer"}

# ─────────────────────────────────────────────────────────
print(f"Indicators: {INDICATORS}")
print(f"Tier 2: buffer={BUFFER_KM}km, HMI top {HMI_PERCENTILE}%")
print(f"NDVI year: {NDVI_YEAR}")

Indicators: []
Tier 2: buffer=100km, HMI top 5%
NDVI year: 2025


## 5. Run the pipeline

In [20]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
import sys
sys.path.append("/content/reference-benchmarking/darukaa_reference_v0.1.0")
from darukaa_reference.config import Config
from darukaa_reference.registry import IndicatorRegistry
from darukaa_reference.indicators import create_default_registry
from darukaa_reference.pipeline import Pipeline

# Build config programmatically (no YAML needed in Colab)
config = Config(
    gee_project=GEE_PROJECT,
    bii_gee_asset="projects/gaurav-singh-007/assets/bii-2020_v2-1-1",
    elevation_band_m=300.0,
    raster_paths={
        "bii": "/content/bii-2020_v2-1-1.tif",
        "iucn_mammals": "projects/darukaa-earth130226/assets/RedList_Mammals_Terrestrial",
        "iucn_birds": "projects/darukaa-earth130226/assets/RedList_Bird_IUCN_Category",
        "kba_global": "projects/darukaa-earth130226/assets/KBA_Global_POL_SEP25",
        "pv_binary": "projects/darukaa-earth-product/assets/biodiversity_India_PV_Binary_2025_Full_Mosaic",
        "msa": "projects/ee-jayankandir/assets/TerrestrialMSA_2015_World"
    },
    output_dir="./output",
    output_format="both",
    enabled_indicators=[],  # empty = all 25
)
# Create registry
registry = create_default_registry()

# Build and run pipeline
pipeline = Pipeline(config, registry)
report = pipeline.run(
    site_path=SITE_FILE,
    output_path="./output/benchmark_scorecard",
)

String: Parameter 'input' is required and may not be null.


## 6. View results

In [21]:
import pandas as pd

# Load scorecard as DataFrame
df = pd.DataFrame(report["scorecard"])

# Display key columns
display_cols = [
    "site_id", "display_name", "pillar", "site_value",
    "tier1_reference", "tier1_intactness",
    "tier2_reference", "tier2_intactness",
    "hedges_g", "permutation_p", "interpretation",
]
existing_cols = [c for c in display_cols if c in df.columns]

print(f"\n{'='*70}")
print(f"BENCHMARK SCORECARD: {len(df)} indicator-site pairs")
print(f"{'='*70}\n")

df[existing_cols].style.format({
    "site_value": "{:.4f}",
    "tier1_reference": "{:.4f}",
    "tier1_intactness": "{:.1%}",
    "tier2_reference": "{:.4f}",
    "tier2_intactness": "{:.1%}",
    "hedges_g": "{:.3f}",
    "permutation_p": "{:.4f}",
}, na_rep="—")


BENCHMARK SCORECARD: 25 indicator-site pairs



,site_id,display_name,pillar,site_value,tier1_reference,tier1_intactness,tier2_reference,tier2_intactness,hedges_g,permutation_p,interpretation
0,Paglam_CR_Demarcation2025_0000,Natural Habitat Extent,1,96.7373,100.0000,96.7%,100.0000,96.7%,—,—,Near-reference condition (96.7% of Tier 2 benchmark)
1,Paglam_CR_Demarcation2025_0000,Natural Land Cover Proportion,1,87.4370,100.0000,87.4%,100.0000,87.4%,—,—,Moderate intactness (87.4% of Tier 2 benchmark)
2,Paglam_CR_Demarcation2025_0000,Landscape Connectivity (CPLAND),1,88.1717,—,—,—,—,—,—,Insufficient data for interpretation
3,Paglam_CR_Demarcation2025_0000,Habitat Loss Rate,1,0.2781,4.1667,100.0%,4.1667,100.0%,—,—,Near-reference condition (100.0% of Tier 2 benchmark)
4,Paglam_CR_Demarcation2025_0000,KBA/IBA Overlap,1,100.0000,—,—,—,—,—,—,Insufficient data for interpretation
5,Paglam_CR_Demarcation2025_0000,Vegetation Structure (NDVI),2,0.6127,0.6523,93.9%,0.8420,72.8%,—,—,Moderate intactness (72.8% of Tier 2 benchmark)
6,Paglam_CR_Demarcation2025_0000,Habitat Health Index (HHI),2,3.4586,5.2426,66.0%,11.6449,29.7%,—,—,Severely degraded (29.7% of Tier 2 benchmark)
7,Paglam_CR_Demarcation2025_0000,Forest Landscape Integrity Index,2,7.5643,9.9680,75.9%,—,—,—,—,Insufficient data for interpretation
8,Paglam_CR_Demarcation2025_0000,Ecosystem Integrity Index,2,0.4678,0.4588,100.0%,0.4823,97.0%,—,—,Near-reference condition (97.0% of Tier 2 benchmark)
9,Paglam_CR_Demarcation2025_0000,EII: Structural Integrity,2,0.6007,0.6286,95.5%,0.6053,99.2%,—,—,Near-reference condition (99.2% of Tier 2 benchmark)


In [ ]:
# Pillar summary
if report.get("pillar_summary"):
    print("\nPILLAR INTACTNESS SUMMARY")
    print("-" * 60)
    for ps in report["pillar_summary"]:
        print(
            f"  Pillar {ps['pillar']}: {ps['pillar_name']:30s} | "
            f"Mean: {ps['mean_intactness']:.1%} | "
            f"Range: [{ps['min_intactness']:.1%}, {ps['max_intactness']:.1%}]"
        )


PILLAR INTACTNESS SUMMARY
------------------------------------------------------------
  Pillar 1: Ecosystem Condition            | Mean: 82.7% | Range: [25.2%, 100.0%]
  Pillar 3: Species/Population Status      | Mean: 93.9% | Range: [93.9%, 93.9%]
  Pillar 4: Threats & Pressures            | Mean: 67.5% | Range: [34.9%, 100.0%]


## 7. Download results

In [ ]:
from google.colab import files

# Download JSON
files.download("./output/benchmark_scorecard.json")

# Download CSV
files.download("./output/benchmark_scorecard.csv")

print("\n✓ Scorecard downloaded (JSON + CSV)")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Scorecard downloaded (JSON + CSV)


---

## (Optional) Add a custom indicator

Adding any new indicator takes just a few lines — no pipeline changes needed.

In [ ]:
# Example: Add CPLAND (connectivity) as a custom indicator

def extract_cpland(geometry, config):
    """
    Your CPLAND extraction logic here.
    Could call Darukaa's internal API, compute from FRAGSTATS, etc.
    """
    # Placeholder — replace with your real implementation
    return {"value": 0.2135, "pixels": None}

registry.register(
    name="cpland",
    display_name="Landscape Connectivity (CPLAND)",
    source_type="api",
    extract_fn=extract_cpland,
    unit="%",
    value_range=(0.0, 100.0),
    citation="McGarigal & Marks (1995). FRAGSTATS. USDA Forest Service.",
    tier2_eligible=True,
    pillar=1,
)

print(f"✓ Registry now has {len(registry)} indicators: {registry.names()}")